# An Interpreter for a Simple Programming Language: Exercise Frame

This notebook is the starting point for the exercise at the end of the chapter *Implementing a Simple
Interpreter*.  It contains the interpreter developed in the notebook `01-Interpreter.ipynb`, extended by
*floating point numbers*, *strings*, and a number of *predefined functions*.  Your task is to extend this
interpreter as follows:

1. Add `for` loops to the interpreter.
2. Add the logical operators `&&` (*and*), `||` (*or*), and `!` (*not*).  The operator `!` should bind
   strongest and the operator `||` should bind weakest.
3. Add user-defined functions:
   - A function should only have access to its parameters and to those variables that are defined
     locally inside the function.
   - A function should always return a value using a `return` statement.  As a `return` statement can
     occur anywhere in a function, use an *exception* to communicate the return value and to transfer
     the control flow out of the function.

The places where you have to add code are marked with the comment `your code here`, both in this notebook and in
the grammar file `sl-frame.lark`.  The notebook runs as it is; only the example programs at the end need your
extensions.

This notebook uses the OCaml port of the parser generator `Larkita`.
The parser generated by `Larkita` returns the abstract syntax tree of a program as a nested tuple, i.e. as a value
of the type `nested` (see `../Chapter-07/01-Calculator.ipynb`).

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml"

The external helper file `AST2Dot.ipynb` is used to represent the nested tuples that serve as our
abstract syntax trees as graphs.

In [ ]:
run_notebook "AST2Dot.ipynb"

## Specification of the Grammar

The grammar is stored in the file `sl-frame.lark`, which the cell below displays.  Compared to the grammar
`sl.lark` of `01-Interpreter.ipynb`, it has the following additions:
- Assignments are described by the syntactical variable `assign`, so that they can also be used inside
  the head of a `for` loop.
- The tokens `FLOAT` and `STRING` describe floating point numbers and string literals.

Add the grammar rules for `for` loops, the logical operators, function definitions, and `return`
statements to the file `sl-frame.lark`.  Remember that the grammar determines the abstract syntax tree:  The string literals of an alternative
form the *head* of the tuple that is returned for this alternative, while all other symbols become its children.
Since the directive `%ignore: "(", ")", "{", "}", ";"` removes the parentheses, the braces, and the semicolon from
all heads, an alternative of the form `"while" "(" bool_expr ")" "{" program "}"` yields the tuple
`("while", test, body)`, where `body` is the tuple `("program", stmnt, ...)`.  As for `if` and `while`, the bodies
of `for` loops and of function definitions have to be enclosed in braces.
If the head given by the literals is not what you want, give the alternative an *alias* `-> "name"`.  An alternative
that consists of a single symbol, like `assign ";"`, returns the tuple of this symbol.  For the logical operators,
introduce the syntactical variables `disjunction`, `conjunction`, and `negation` in order to encode the precedences.

In [ ]:
cat "sl-frame.lark"

We generate the parser.  `Larkita` always generates a *canonical LR(1) parser*.  A description of the states of the
LR(1) automaton is written into the file `sl.out`.

In [ ]:
let parser = Larkita.open_ ~out_file:"sl.out" "sl-frame.lark"

## The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple.  The leaves of this tree are tokens, which are
represented as pairs of the name of the token and its text:  `("INT", "42")`, `("FLOAT", "1.5")`,
`("ID", "x")`, and `("STRING", "\"abc\"")`.  Note that the text of a string literal includes its double quotes.

The values of our variables are integers, floating point numbers, or strings.  They are values of the type
`value` from `../lib/nested.ml`:  `Int n`, `Float x`, and `Str s`.

The shapes of the tuples that are produced by the grammar above are the following:

| construct              | nested tuple                                       |
|:-----------------------|:---------------------------------------------------|
| program, body          | `("program", stmnt, ...)`                          |
| assignment             | `(":=", ("ID", name), expr)`                       |
| expression statement   | `("expr", expr)`                                   |
| conditional, loop      | `("if", test, body)`, `("while", test, body)`      |
| binary operator        | `(op, lhs, rhs)`, for example `("+", lhs, rhs)`    |
| function call          | `("call", ("ID", name), arg, ...)`                 |

For example, the following program yields the tuple shown below:

In [ ]:
Larkita.parse parser {|x := 1.5 * y; print("x = ", x);|}

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed into
an abstract syntax tree, which is then displayed as a graph.

In [ ]:
let parse (file_name : string) : Graphviz.t =
  let program = read_file file_name in
  print_endline program;
  let ast = Larkita.parse parser program in
  print_endline (Nested.to_string ast);
  tuple2dot ast

In [ ]:
Graphviz.display (parse "sum.sl")

## The Runtime System

A `return` statement can occur anywhere inside a function.  Therefore, it is implemented by raising an
exception `Return_value`, which carries the value that is returned.  The call of the function
catches this exception.

In [ ]:
exception Return_value of value

The hash table `functions` stores the user-defined functions.  It maps the name of a function to a pair
consisting of the list of its parameters and its body.

In [ ]:
let functions : (string, string list * nested) Hashtbl.t = Hashtbl.create 16

The function `execute_tuple` executes a list of statements.  The hash table `values` maps the names of
variables to their values.

As the functions below call each other, the next cell defines *forward references* to them (see
`01-Interpreter.ipynb`).

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

type env = (string, value) Hashtbl.t

let execute_ref       : (nested -> env -> unit) ref          = ref not_yet_defined
let evaluate_bool_ref : (nested -> env -> bool) ref          = ref not_yet_defined
let evaluate_ref      : (nested -> env -> value) ref         = ref not_yet_defined
let call_builtin_ref  : (string -> value list -> value) ref = ref not_yet_defined

let execute_tuple (statementList : nested list) (values : env) : unit =
  List.iter (fun stmnt -> !execute_ref stmnt values) statementList

The function `execute` matches recursively against the shape of the nested tuple that represents a
statement.

In [ ]:
let rec execute (stmnt : nested) (values : env) : unit =
  match stmnt with
  | Node (Leaf (Str "program") :: sl) -> execute_tuple sl values
  | Node [ Leaf (Str ":="); Node [ Leaf (Str "ID"); Leaf (Str var) ]; value ] -> Hashtbl.replace values var (!evaluate_ref value values)
  | Node [ Leaf (Str "expr"); expr ] -> ignore (!evaluate_ref expr values)
  | Node [ Leaf (Str "if"); test; stmnt ] -> if !evaluate_bool_ref test values then execute stmnt values
  | Node [ Leaf (Str "while"); test; stmnt ] ->
      while !evaluate_bool_ref test values do
        execute stmnt values
      done
  (* your code here: for loops *)
  (* your code here: function definitions *)
  (* your code here: return statements (raise a Return_value exception) *)
  | _ -> failwith (Nested.to_string stmnt ^ " unexpected")
;;
execute_ref := execute

The function `number` checks that a value is a number.  It is needed since our values may also be
strings, but the arithmetical operators, the comparison operators `<`, `<=`, `>`, `>=`, and the
mathematical functions are only defined for numbers.

In [ ]:
let number (v : value) : float =
  match v with
  | Int n -> float_of_int n
  | Float x -> x
  | Str s -> failwith (Nested.quote s ^ " is not a number")
  | Bool _ -> failwith "a Boolean value is not a number"

The function `evaluate_bool` evaluates the test of a conditional or of a loop and returns a Boolean
value.

In [ ]:
(* == and != compare numbers by their values, and strings with strings *)
let equal (a : value) (b : value) : bool =
  match a, b with
  | Str x, Str y -> x = y
  | Str _, _ | _, Str _ -> false
  | _ -> number a = number b

let evaluate_bool (expr : nested) (values : env) : bool =
  let eval e = !evaluate_ref e values in
  match expr with
  | Node [ Leaf (Str "=="); lhs; rhs ] -> equal (eval lhs) (eval rhs)
  | Node [ Leaf (Str "!="); lhs; rhs ] -> not (equal (eval lhs) (eval rhs))
  | Node [ Leaf (Str "<="); lhs; rhs ] -> number (eval lhs) <= number (eval rhs)
  | Node [ Leaf (Str ">="); lhs; rhs ] -> number (eval lhs) >= number (eval rhs)
  | Node [ Leaf (Str "<"); lhs; rhs ] -> number (eval lhs) <  number (eval rhs)
  | Node [ Leaf (Str ">"); lhs; rhs ] -> number (eval lhs) >  number (eval rhs)
  (* your code here: the logical operators &&, ||, and ! *)
  | _ -> failwith (Nested.to_string expr ^ " unexpected")
;;
evaluate_bool_ref := evaluate_bool

The function `evaluate` computes the value of an expression.  A call of a function is either a call of a
user-defined function or a call of a predefined function.
The leaves of the abstract syntax tree are tokens:  A token of type `INT` or `FLOAT` is converted into a number,
the text of a token of type `STRING` is stripped of its double quotes, and the value of a variable, i.e. of a token
of type `ID`, is looked up in `values`.

The function `arith` applies an arithmetical operator:  the result is an integer if both arguments are integers,
and a floating point number otherwise.  `floor_mod` and `floor_fmod` compute the remainder of a division that rounds
down, so that the remainder has the sign of the divisor.

In [ ]:
let arith (int_op : int -> int -> int) (float_op : float -> float -> float) (a : value) (b : value) : value =
  match a, b with
  | Int x, Int y -> Int (int_op x y)
  | _ -> Float (float_op (number a) (number b))

let floor_mod (x : int) (y : int) : int = ((x mod y) + y) mod y

let floor_fmod (x : float) (y : float) : float = x -. y *. Float.floor (x /. y)

let rec evaluate (expr : nested) (values : env) : value =
  let eval e = evaluate e values in
  match expr with
  | Node [ Leaf (Str "INT"); Leaf (Str digits) ] -> Int (int_of_string digits)
  | Node [ Leaf (Str "FLOAT"); Leaf (Str digits) ] -> Float (float_of_string digits)
  | Node [ Leaf (Str "STRING"); Leaf (Str text) ] -> Str (String.sub text 1 (String.length text - 2))
  | Node [ Leaf (Str "ID"); Leaf (Str name) ] -> Hashtbl.find values name
  | Node [ Leaf (Str "+"); lhs; rhs ] -> arith ( + ) ( +. ) (eval lhs) (eval rhs)
  | Node [ Leaf (Str "-"); lhs; rhs ] -> arith ( - ) ( -. ) (eval lhs) (eval rhs)
  | Node [ Leaf (Str "*"); lhs; rhs ] -> arith ( * ) ( *. ) (eval lhs) (eval rhs)
  | Node [ Leaf (Str "/"); lhs; rhs ] -> Float (number (eval lhs) /. number (eval rhs))
  | Node [ Leaf (Str "%"); lhs; rhs ] -> arith floor_mod floor_fmod (eval lhs) (eval rhs)
  | Node (Leaf (Str "call") :: Node [ Leaf (Str "ID"); Leaf (Str f_name) ] :: _args) when Hashtbl.mem functions f_name ->
      (* your code here: call a user-defined function
          - evaluate the arguments,
          - execute the body with a new hash table that contains only the parameters,
          - the body returns its value by raising a Return_value exception. *)
      failwith "NotImplementedError: calls of user-defined functions"
  | Node (Leaf (Str "call") :: Node [ Leaf (Str "ID"); Leaf (Str f_name) ] :: args) -> !call_builtin_ref f_name (List.map eval args)
  | _ -> failwith (Nested.to_string expr ^ " unexpected")
;;
evaluate_ref := evaluate

The function `call_builtin` implements the predefined functions of our language.  The function `print`
prints all of its arguments without separators between them.

In [ ]:
let value_to_string (v : value) : string =
  match v with
  | Int n -> string_of_int n
  | Float x -> float_repr x
  | Bool b -> string_of_bool b
  | Str s -> s

let call_builtin (f_name : string) (args : value list) : value =
  match f_name, args with
  | "read", _ ->
      let s = input "Please enter a number: " in
      if String.contains s '.' || String.contains s 'e' || String.contains s 'E' then Float (float_of_string s)
      else Int (int_of_string s)
  | "print", _ ->
      print_endline (String.concat "" (List.map value_to_string args));
      Int 0
  | "sqrt", x :: _ -> Float (Float.sqrt (number x))
  | "exp", x :: _ -> Float (Float.exp (number x))
  | "ln", x :: _ -> Float (Float.log (number x))
  | "sin", x :: _ -> Float (Float.sin (number x))
  | "cos", x :: _ -> Float (Float.cos (number x))
  | "tan", x :: _ -> Float (Float.tan (number x))
  | "arctan", x :: _ -> Float (Float.atan (number x))
  | _ -> failwith (Printf.sprintf "function %s is unknown" f_name)
;;
call_builtin_ref := call_builtin

## Putting Everything Together

The function `main` reads a program from a file, parses it, and executes it.  If `show_ast` is `true`, the
abstract syntax tree is displayed first.

In [ ]:
let main ?(show_ast : bool = false) (file_name : string) : unit =
  let program = read_file file_name in
  let ast = Larkita.parse parser program in
  if show_ast then Graphviz.display (tuple2dot ast);
  Hashtbl.reset functions;
  execute ast (Hashtbl.create 16)

The program `sum.sl` only uses features that are already implemented.

In [ ]:
cat "sum.sl"

In [ ]:
simulate_input [ "10" ];;
main "sum.sl"

## Testing Your Extensions

The following programs need your extensions.  The program `factorial.sl` uses user-defined functions and
a `for` loop.

In [ ]:
cat "factorial.sl"

In [ ]:
main "factorial.sl"

In [ ]:
cat "e.sl"

In [ ]:
main "e.sl"

In [ ]:
cat "pi.sl"

In [ ]:
main "pi.sl"

In [ ]:
cat "solve.sl"

In [ ]:
main "solve.sl"

In [ ]:
cat "sum-for.sl"

In [ ]:
simulate_input [ "10" ];;
main "sum-for.sl"